In [ ]:
# === Notebook path configuration ===
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
if CWD.name == "diagnostic":
    DIAGNOSTIC_DIR = CWD
elif (CWD / "diagnostic").is_dir():
    DIAGNOSTIC_DIR = CWD / "diagnostic"
else:
    DIAGNOSTIC_DIR = next((p for p in [CWD, *CWD.parents] if p.name == "diagnostic"), CWD.parent)

if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))

import os
import re
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.dates import DateFormatter


class ObservedTCIPlotter:
    def __init__(self, tci_dir, regional_dir=None, fig_dir=None,
                 cmap='viridis', vmin=None, vmax=None):
        self.tci_dir = tci_dir
        self.regional_dir = regional_dir or tci_dir
        self.fig_dir = fig_dir or os.path.join(tci_dir, "figures")
        os.makedirs(self.fig_dir, exist_ok=True)

        self.cmap = cmap
        self.vmin = vmin
        self.vmax = vmax

    # ========== UTILS ==========

    def _extract_window_time(self, tag):
        match = re.search(r'\d{4}-\d{2}', tag)
        if match:
            return match.group(0)
        return tag

    
        # ========== MAP PLOTS ==========
    def plot_window_grid_map(self, sort_windows=True, projection="Robinson"):
        import matplotlib.gridspec as gridspec
    
        files = [
            f for f in os.listdir(self.tci_dir)
            if f.startswith("TCI_obs_window_") and f.endswith(".nc")
        ]
    
        if not files:
            print("[WARN] No window files found.")
            return
    
        # Sort by label like D01–15, D16–30, etc.
        if sort_windows:
            files.sort(key=lambda f: f.replace(".nc", "").split("_")[-1])
    
        nrows = len(files)
        fig = plt.figure(figsize=(12, 2.5 * nrows))
        gs = gridspec.GridSpec(nrows, 1, hspace=0.3)
    
        for i, fname in enumerate(files):
            path = os.path.join(self.tci_dir, fname)
            ds = xr.open_dataset(path)
            if 'TCI' not in ds:
                continue
    
            da = ds['TCI'].squeeze()
    
            ax = fig.add_subplot(gs[i], projection=ccrs.Robinson() if projection == "Robinson" else ccrs.PlateCarree())
            ax.set_global()

            p = da.plot.pcolormesh(
                ax=ax,
                transform=ccrs.PlateCarree(),
                cmap=self.cmap,
                vmin=self.vmin,
                vmax=self.vmax,
                add_colorbar=False,
                add_labels=False
            )
            ax.coastlines()
            ax.set_title(fname.replace("TCI_obs_window_", "").replace(".nc", ""), fontsize=10)
    
        # Add single shared colorbar
        cbar_ax = fig.add_axes([0.25, 0.05, 0.5, 0.02])
        cbar = fig.colorbar(p, cax=cbar_ax, orientation='horizontal')
        cbar.set_label("TCI (unitless)")
    
        save_path = os.path.join(self.fig_dir, "TCI_obs_window_grid.png")
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"[SAVED] {save_path}")
        plt.close(fig)

    def plot_tci_map(self, filename, title=None, save=True):
        filepath = os.path.join(self.tci_dir, filename)
        if not os.path.exists(filepath):
            print(f"[WARN] Missing TCI file: {filepath}")
            return

        ds = xr.open_dataset(filepath)
        if 'TCI' not in ds:
            raise KeyError(f"[ERROR] 'TCI' variable not found in {filepath}")

        da = ds['TCI'].squeeze()

        fig = plt.figure(figsize=(10, 5))
        ax = plt.axes(projection=ccrs.Robinson())
        ax.set_global()
        im = da.plot.pcolormesh(
            ax=ax,
            transform=ccrs.PlateCarree(),
            cmap=self.cmap,
            vmin=self.vmin,
            vmax=self.vmax,
            add_colorbar=False,
            add_labels=False
        )
        cbar = plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8)
        cbar.set_label('TCI (unitless)')

        ax.coastlines()
        ax.add_feature(cfeature.BORDERS, linewidth=0.5)
        ax.set_title(title or filename.replace('.nc', ''), fontsize=12)

        if save:
            save_name = filename.replace('.nc', '.png')
            save_path = os.path.join(self.fig_dir, save_name)
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"[SAVED] {save_path}")

        plt.close(fig)

    def plot_all_maps(self):
        for fname in sorted(os.listdir(self.tci_dir)):
            if fname.startswith("TCI_obs_window_") and fname.endswith(".nc"):
                window = fname.split("_window_")[-1].replace(".nc", "")
                self.plot_tci_map(fname, title=f"TCI (Obs) – {window}")

    # ========== REGIONAL MEAN LOADER ==========

    def load_regional_means_with_error(self, directory, prefix="TCI_obs_regional_mean_"):
        files = sorted([
            f for f in os.listdir(directory)
            if f.startswith(prefix) and f.endswith(".nc")
        ])
        region_data = {}

        for fname in files:
            tag = fname.replace(prefix, "").replace(".nc", "")
            time_str = self._extract_window_time(tag)
            ds = xr.open_dataset(os.path.join(directory, fname))

            for var in ds.data_vars:
                base = var.replace("TCI_", "").replace("_stderr", "")
                key = f"{base}_stderr" if "stderr" in var else base

                if base not in region_data:
                    region_data[base] = []

                region_data[base].append({
                    "time": time_str,
                    "value" if "stderr" not in key else "stderr": ds[var].values.item()
                })

        df_dict = {}
        for region, records in region_data.items():
            df = pd.DataFrame(records).groupby("time", as_index=False).first()
            df = df.sort_values("time")
            df_dict[region] = df

        return df_dict

    # ========== REGIONAL TIME SERIES ==========

    def plot_regional_time_series(
        self,
        obs_df_dict,
        model_df_dicts=None,
        model_labels=None,
        show_errorbars=True,
        use_datetime_x=True,
        title="TCI Regional Means: Observations vs Models",
        filename="TCI_obs_vs_model_regional_means.png"
    ):
        fig, ax = plt.subplots(figsize=(11, 6))

        # Plot OBSERVATIONS
        for region, df in obs_df_dict.items():
            x = pd.to_datetime(df["time"]) if use_datetime_x else df["time"]
            y = df["value"]
            yerr = df["stderr"] if show_errorbars and "stderr" in df else None
            ax.errorbar(x, y, yerr=yerr, label=f"{region} (OBS)", fmt='-o')

        # Plot MODEL overlays
        if model_df_dicts:
            for model_label, model_dict in zip(model_labels, model_df_dicts):
                for region, df in model_dict.items():
                    x = pd.to_datetime(df["time"]) if use_datetime_x else df["time"]
                    y = df["value"]
                    yerr = df["stderr"] if show_errorbars and "stderr" in df else None
                    ax.errorbar(
                        x, y, yerr=yerr, linestyle='--', marker='x',
                        label=f"{region} ({model_label})"
                    )

        ax.set_title(title)
        ax.set_ylabel("TCI (unitless)")
        ax.set_xlabel("Time")
        ax.grid(True, linestyle='--', alpha=0.4)
        ax.legend(loc='best', fontsize=9)

        if use_datetime_x:
            ax.xaxis.set_major_formatter(DateFormatter('%Y-%m'))

        plt.xticks(rotation=45)
        plt.tight_layout()

        save_path = os.path.join(self.fig_dir, filename)
        plt.savefig(save_path, dpi=150)
        print(f"[SAVED] {save_path}")
        plt.close(fig)

    # ========== FULL PIPELINE ==========

    def run_all_plots(
        self,
        model_dirs_labels=None,  # List of tuples: (model_dir, label)
        show_errorbars=True
    ):
        print("[INFO] Plotting all maps...")
        self.plot_all_maps()

        print("[INFO] Loading observation regional means...")
        obs_df = self.load_regional_means_with_error(self.regional_dir)

        model_df_list = []
        model_labels = []
        if model_dirs_labels:
            for model_dir, label in model_dirs_labels:
                model_df = self.load_regional_means_with_error(model_dir, prefix="TCI_model_regional_mean_")
                model_df_list.append(model_df)
                model_labels.append(label)

        print("[INFO] Plotting regional time series...")
        self.plot_regional_time_series(
            obs_df_dict=obs_df,
            model_df_dicts=model_df_list,
            model_labels=model_labels,
            show_errorbars=show_errorbars,
            use_datetime_x=True,
        )


In [5]:
if __name__ == "__main__":
    top_path = "/pscratch/sd/z/zhan391/e3sm_dart"
    out_path = "/pscratch/sd/z/zhan391/e3sm_dart/diag_dart"
    fig_path = "/global/homes/z/zhan391/analysis/diagnostic/figures"
    landmask_file = os.path.join(out_path, "landmask_1x1.nc")

    # Labeled time windows
    years = [2012]   
    windows_dict = {
        "D1-15":   ("2012-01-01", "2012-01-15"),
        "D16-30":  ("2012-01-16", "2012-01-30"),
        "D31-45":  ("2012-01-31", "2012-02-14"),
        "D46-60":  ("2012-02-15", "2012-03-01"),
        "2012-01": ("2012-01-01", "2012-01-31"),
        "2012-02": ("2012-02-01", "2012-02-29"),
    }

    
    years = [2011]   
    windows_dict = {
        "D1-15-2011":   ("2011-12-01", "2011-12-15"),
        "D16-30-2011":  ("2011-12-16", "2011-12-31"),
        "2011-12": ("2011-12-01", "2011-12-31"),
    }    
    
    key = "H2OSOI_LHFLX"
    tci_dir = f'{out_path}/obs_tci'
    for window in windows_dict.keys():
        plotter = ObservedTCIPlotter(
            tci_dir=tci_dir,
            regional_dir=tci_dir,
            fig_dir=fig_path,
            cmap="magma",
            vmin=-5,
            vmax=5,
        )
        plotter.plot_window_grid_map()

         #plotter.run_all_plots(
         #    model_dirs_labels=[
         #       ("./CTRLEN10_tci_output", "CTRLEN10"),
         #       ("./CAPTEN20_tci_output", "CAPTEN20")
         #   ],
         #   show_errorbars=True
         #)

        #savefile = os.path.join(fig_path, f"TCI_diff_{key}_{window}.pdf")
        #plotter.plot_tci_map(compare=False, savepath=None) #savefile)


[SAVED] /global/homes/z/zhan391/analysis/diagnostic/figures/TCI_obs_window_grid.png
[SAVED] /global/homes/z/zhan391/analysis/diagnostic/figures/TCI_obs_window_grid.png
[SAVED] /global/homes/z/zhan391/analysis/diagnostic/figures/TCI_obs_window_grid.png
